### Config

In [1]:
import tiktoken
from torch.utils.data import Dataset, DataLoader

In [2]:
DATA_PATH = "data/short_story.txt"
CONTEXT_SIZE = 8
BATCH_SIZE = 32
SHUFFLE = True

### Prepare tokenized data

In [3]:
text = None
with open(DATA_PATH, 'r') as f:
    text = f.read()

In [4]:
tokenizer = tiktoken.get_encoding("gpt2")
tokens = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
tokens[:10]

[40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138]

### Simulate casual attention mask

In [5]:
training_pairs = []
for i in range(len(tokens[:200])):
    x = tokens[i:CONTEXT_SIZE+i]
    y = tokens[CONTEXT_SIZE+i]
    training_pairs.append((x, y))
for (x,y) in training_pairs[:10]:
    print(tokenizer.decode(x), " -> ", tokenizer.decode([y]))

I HAD always thought Jack Gis  ->  burn
 HAD always thought Jack Gisburn  ->   rather
AD always thought Jack Gisburn rather  ->   a
 always thought Jack Gisburn rather a  ->   cheap
 thought Jack Gisburn rather a cheap  ->   genius
 Jack Gisburn rather a cheap genius  ->  --
 Gisburn rather a cheap genius--  ->  though
isburn rather a cheap genius--though  ->   a
burn rather a cheap genius--though a  ->   good
 rather a cheap genius--though a good  ->   fellow


### Pytorch Dataset & DataLoader
Why not just shifting sequences by 1: 
- shifting sequences by 1 causes lots of redundant overlaps where LLM is trained on almost the same sequence
- => better approach is to create no / low overlapping sampling
- we choose no overlaps for simplicity
-  stride parameter could be added to control the overlap: `overlap = T - stride`

In [16]:
class TrainDataset(Dataset):
    def __init__(self, tokens: list[int], context_size: int, stride: int):
        self.tokens = tokens
        self.context_size = context_size

    def __getitem__(self, index: int) -> tuple[list[int], list[int]]:
        x = tokens[i:self.context_size+i]
        y = tokens[i+1:self.context_size+i+1]
        return (x, y)
        
    def __len__(self) -> int:
        # last sequence is reserved as prediction target y
        return len(self.tokens) - self.context_size

In [10]:
train_data = TrainDataset(tokens=tokens, context_size=CONTEXT_SIZE)

In [13]:
loader = DataLoader(train_data, batch_size=BATCH_SIZE, 
                    shuffle=SHUFFLE, drop_last=True)

In [14]:
for x, y in loader:
    print('.')

.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
